# Stage 14 — Owner-only Hugging Face adapter release

**This notebook NEVER starts training or GPU inference.** It prepares the exact source-hashed original 40-step adapter-only release. All publication switches start **OFF**.

**Do not make the adapter public unless you personally reviewed the original model's CC BY-NC 4.0 license, Cohere acceptable-use policy, your rights to release the adapter and synthetic aggregate results, and the documented safety regression.** The upstream base model remains gated and is not included.

Our source ZIP SHA256 must equal `dcefc5b0babf708aa1b48ac0349ddb3bc4745405c72e9636c5860d7719d8f0e9`. It is the owner's original `policyproof_align_native40.zip`, **not** the curated 78-example training ZIP or Stage 13 results ZIP.


## 1. Fetch latest repo (CPU runtime; no GPU needed)

If Colab asks to restart after dependency installation, restart the session and run this setup cell again. No model weights are downloaded in this notebook.


In [ ]:
from pathlib import Path
import os, sys, subprocess
ROOT=Path('/content/PolicyProof-AI')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/betterlifestyle04-commits/PolicyProof-AI.git',str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'],check=True)
os.chdir(ROOT)
print('REPO_READY',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU/optimizer use: NONE')


## 2. Upload the ORIGINAL, private, 40-step adapter ZIP

This checks the original SHA before any publication preparation. Do **not** upload a Hugging Face API token as a file or paste it into a notebook source cell.


In [ ]:
from google.colab import files
import hashlib
ORIGINAL=Path('/content/policyproof_align_native40.zip')
EXPECTED='dcefc5b0babf708aa1b48ac0349ddb3bc4745405c72e9636c5860d7719d8f0e9'
if not ORIGINAL.is_file():
    os.chdir('/content')
    uploaded=files.upload()
    assert set(uploaded)=={'policyproof_align_native40.zip'}, 'Upload exactly the original native40 adapter ZIP'
assert hashlib.sha256(ORIGINAL.read_bytes()).hexdigest()==EXPECTED, 'Wrong source ZIP: STOP'
os.chdir(ROOT)
print('ORIGINAL_OWNER_ADAPTER_SHA_VERIFIED: TRUE')


## 3. Offline release preview — still NOT published

This checks the three source files, verifies the LoRA and training receipt, and produces a model-card package containing only adapter weights, config, aggregate development metrics and a source-hashed manifest. It includes no gated Tiny Aya base weights or private raw model outputs. You can download and examine the preview before continuing.


In [ ]:
PREVIEW=Path('/content/policyproof-native40-HF-PREVIEW-NOT_PUBLISHED.zip')
if PREVIEW.exists():
    print('Existing preview found. Inspect its checksum and do not overwrite.')
else:
    subprocess.run([sys.executable,'-m','scripts.owner_native40_hf_release',
        'preview','--archive',str(ORIGINAL),'--preview-zip',str(PREVIEW)],
        cwd=ROOT,check=True)
assert PREVIEW.is_file()
print('PREVIEW_ONLY — PUBLICATION HAS NOT OCCURRED')
print('SHA256',hashlib.sha256(PREVIEW.read_bytes()).hexdigest())
files.download(str(PREVIEW))


## 4. Owner decision — do not skip the license review

Open [Tiny Aya Global's license and gated-access conditions](https://huggingface.co/CohereLabs/tiny-aya-global), review [our detailed model card](https://github.com/betterlifestyle04-commits/PolicyProof-AI/blob/main/docs/HF_NATIVE40_ADAPTER_MODEL_CARD.md), and inspect the downloaded preview. Review whether **you have the rights** to distribute the adapter and aggregate results publicly as noncommercial research. The adapter worsened strict unsupported-question abstention from 4/9 to 2/9 on the small development set. It must not be promoted as a safety-improved production model.

**Only if you personally approve**, change all three variables below to True. Nothing in the notebook or GitHub may grant these approvals on your behalf.


In [ ]:
PUBLIC_ADAPTER_RELEASE_APPROVED = False
UPSTREAM_LICENSE_AND_COHERE_AUP_REVIEWED = False
AGGREGATE_RESEARCH_RELEASE_RIGHTS_APPROVED = False

def owner_approved():
    return all([
        PUBLIC_ADAPTER_RELEASE_APPROVED,
        UPSTREAM_LICENSE_AND_COHERE_AUP_REVIEWED,
        AGGREGATE_RESEARCH_RELEASE_RIGHTS_APPROVED,
    ])

print('PUBLIC RELEASE APPROVED:', owner_approved())
print('No Hugging Face write action in this cell.')


## 5. Optional secure Hugging Face login

Only run the authentication widget if you personally approved the public release above. Log in with **the Hugging Face account that owns the published dataset** (previously `shalnark01`). Use a write-capable token in Hugging Face's official widget. **Never paste your token into this chat, notebook source, GitHub, or a public file.**


In [ ]:
if owner_approved():
    from huggingface_hub import notebook_login
    notebook_login(skip_if_logged_in=False)
    print('Complete the secure login widget, then continue to Step 6.')
else:
    print('Hugging Face login not requested: rights approval is still OFF.')


## 6. Explicit public release (OFF by default)

This step would **create a public, ungated adapter-only model repository and public model collection** under the authenticated HF namespace. It does not publish the gated Tiny Aya base weights. No preexisting repository is overwritten. The result is treated as complete only after anonymous readback verifies the model and all uploaded file hashes, plus its collection membership.

**Set `RUN_PUBLIC_HF_RELEASE=True` only after owner approval and secure login.** If publication fails midway, review the partial Hugging Face repository manually before retrying; do not invent a successful URL.


In [ ]:
RUN_PUBLIC_HF_RELEASE = False  # owner must explicitly opt in
HF_NAMESPACE = 'shalnark01'  # must match the authenticated Hugging Face username
if RUN_PUBLIC_HF_RELEASE:
    assert owner_approved(), 'Rights/license/public release not approved'
    subprocess.run([sys.executable,'-m','scripts.owner_native40_hf_release',
        'publish','--archive',str(ORIGINAL),
        '--namespace',HF_NAMESPACE,
        '--confirmed-source-sha256',EXPECTED,
        '--approve-public','--approve-license','--approve-outputs'],
        cwd=ROOT,check=True)
else:
    print('PUBLICATION OFF: model repo and collection have NOT been created by this notebook.')


## 7. Finish Lexsi's official submission manually

When the upload command has *actually* verified a public model collection, copy **the exact model collection URL it printed**. Use the already-public dataset collection, the GitHub repo, and the updated research write-up. Open [Lexsi submission](https://hackathon.lexsi.ai/submission/), enter your own registered email and the under-500-word abstract, and save the organizer confirmation.

**Deadline: 15 November 2026, 11:59 PM AoE.** A public model upload is **not** an official Lexsi submission. If Hugging Face rights approval cannot be obtained, do not publish the adapter; ask Lexsi whether a benchmark-only dataset/report can meet the public model-collection field instead.
